# 🧠 DSA Lab 12: Hashing: Hash Functions, Collisions and Hash Tables
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 12 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: Hash Functions

A **hash function** converts a **key** (number, string, tuple…) into an **index** of an array called the **hash table** (or bucket array). The table stores the value at that index.

**Key → hash → index**

```text
key "Ali"  --hash function-->  2  --> table[2] = ("Ali", 3.4)

index:   0        1        2              3        4
      [     ] [     ] [Ali: 3.4] [     ] [     ]
```

### A Simple Hash Function for Integers

The **division method**: `h(x) = x % m`, where m is the table size. With m = 10, the values 11, 12, 13, 14, 15 go to indexes 1, 2, 3, 4, 5 (exactly the example in your theory slides). A **prime** table size usually spreads keys better.

In [ ]:
def h(x, m=10):
    return x % m

for key in [11, 12, 13, 14, 15, 25, 35]:
    print(key, "->", h(key))          # 15, 25, 35 all go to 5: collisions!

### Hashing Strings

In [ ]:
def bad_hash(s, m):                    # sum of character codes
    return sum(ord(c) for c in s) % m

def poly_hash(s, m, base=31):          # polynomial rolling hash (used in Java)
    h = 0
    for c in s:
        h = (h * base + ord(c)) % m
    return h

for w in ["listen", "silent", "enlist"]:
    print(f"{w:<7} bad={bad_hash(w, 101):>3}  poly={poly_hash(w, 101):>3}")

Anagrams have the same characters, so the sum-of-codes hash puts them all in the same bucket. The polynomial hash uses the **position** of each character too, so it spreads them out.

| Property of a good hash function | Why |
|---|---|
| Deterministic | The same key must always give the same index |
| Fast to compute | Hashing must be O(1) (or O(length of key)) |
| Uniform distribution | Spreads keys evenly to minimise collisions |
| Uses the whole key | Small differences in keys should give different indexes |

In [ ]:
print(hash(42), hash("Okara") % 100, hash((1, 2)) % 100)
try:
    hash([1, 2])
except TypeError as e:
    print("Lists cannot be keys:", e)

Only **immutable** objects (int, str, tuple of immutables) can be dictionary keys or set items. If a key could change after insertion, its hash would change and it could never be found again.

## 💥 Part B: Collisions and How to Resolve Them

A **collision** happens when two different keys hash to the same index. Collisions are **unavoidable** (there are more possible keys than indexes, the **pigeonhole principle**), so every hash table needs a strategy.

### Strategy 1: Separate Chaining

Each slot holds a **list (chain)** of all key-value pairs that hash there. To search, hash the key and scan only that short chain.

**Separate chaining**

```text
m = 5, keys: 15, 22, 7, 30, 12   (h = key % 5)

0: [15] -> [30]
1:
2: [22] -> [7] -> [12]
3:
4:
```

### Strategy 2: Open Addressing (Linear Probing)

All items live in the array itself. If the slot is taken, try the **next** slot (`(i + 1) % m`), and keep going until an empty slot is found. Search follows the same path. Deleting requires a special **tombstone** marker so later searches do not stop early.

**Linear probing**

```text
m = 7, insert 10, 17, 24, 5   (h = key % 7)
10 -> 3                        [ _, _, _, 10, _, _, _ ]
17 -> 3 taken, try 4           [ _, _, _, 10, 17, _, _ ]
24 -> 3, 4 taken, try 5        [ _, _, _, 10, 17, 24, _ ]
5  -> 5 taken, try 6           [ _, _, _, 10, 17, 24, 5 ]
```

|  | Separate chaining | Linear probing |
|---|---|---|
| Where items live | Lists hanging from each slot | Inside the array |
| Load factor can exceed 1? | Yes | No (must stay below ~0.7) |
| Deletion | Easy | Needs tombstones |
| Cache performance | Poorer (pointers) | Excellent (contiguous) |
| Problem | Long chains if the hash is bad | Clustering: long runs of filled slots |

### Load Factor and Rehashing

The **load factor** α = n / m (items ÷ slots). As α grows, chains get longer (or probe sequences get longer) and operations slow down. When α passes a threshold (e.g., 0.75), the table **doubles its size** and **re-inserts every item** with the new m. Like the dynamic array in Lab 03, this keeps operations **O(1) amortised**.

| Operation | Average | Worst (all keys collide) |
|---|---|---|
| Search | O(1) | O(n) |
| Insert | O(1) amortised | O(n) |
| Delete | O(1) | O(n) |

## 🛠️ Part C: Build Your Own Hash Map (Separate Chaining)

In [ ]:
class HashMap:
    def __init__(self, capacity=8):
        self.buckets = [[] for _ in range(capacity)]
        self.size = 0

    def _index(self, key):
        return hash(key) % len(self.buckets)

    def put(self, key, value):
        bucket = self.buckets[self._index(key)]
        for pair in bucket:
            if pair[0] == key:             # key exists: update
                pair[1] = value
                return
        bucket.append([key, value])        # new key
        self.size += 1
        if self.size / len(self.buckets) > 0.75:
            self._resize()

    def get(self, key, default=None):
        for k, v in self.buckets[self._index(key)]:
            if k == key:
                return v
        return default

    def remove(self, key):
        bucket = self.buckets[self._index(key)]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self.size -= 1
                return v
        raise KeyError(key)

    def _resize(self):
        old = self.buckets
        self.buckets = [[] for _ in range(2 * len(old))]
        self.size = 0
        for bucket in old:
            for k, v in bucket:
                self.put(k, v)             # re-hash into the bigger table

    def __len__(self):
        return self.size

cgpa = HashMap()
for name, g in [("Ali", 3.4), ("Sara", 3.9), ("Hamza", 3.1), ("Zoya", 3.7), ("Bilal", 2.9),
                ("Iqra", 3.5), ("Omar", 3.2)]:
    cgpa.put(name, g)
cgpa.put("Ali", 3.6)                       # update
print(cgpa.get("Ali"), cgpa.get("Nobody", "not found"), len(cgpa), "buckets:", len(cgpa.buckets))
print("longest chain:", max(len(b) for b in cgpa.buckets))

## 🐍 Part D: Hashing Tools in Python

In [ ]:
from collections import Counter, defaultdict

votes = ["Ali", "Sara", "Hamza", "Ali", "Hamza", "Ali"]   # class representative election
tally = Counter(votes)                      # frequency counting in one line
print(tally, tally.most_common(1))

by_city = defaultdict(list)                 # missing keys get a default empty list
for name, city in [("Ali", "Okara"), ("Sara", "Lahore"), ("Hamza", "Okara")]:
    by_city[city].append(name)
print(dict(by_city))

bscs, bsit = {"Ali", "Sara", "Hamza"}, {"Sara", "Usman"}
print("both:", bscs & bsit, "| only BSCS:", bscs - bsit, "| any:", bscs | bsit)

marks = {"Ali": 81, "Sara": 92}
print(marks.get("Zoya", 0), "Ali" in marks, list(marks.items()))

### Cryptographic Hashes (Security)

In [ ]:
import hashlib
pw = "okara123"
print(hashlib.sha256(pw.encode()).hexdigest())
print(hashlib.sha256("okara124".encode()).hexdigest())   # tiny change -> totally different

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <unordered_map>
#include <unordered_set>
#include <vector>
#include <list>
using namespace std;

int main() {
    unordered_map<string, double> cgpa;      // hash table (separate chaining)
    cgpa["Ali"] = 3.4; cgpa["Sara"] = 3.9;
    cout << cgpa["Sara"] << " " << cgpa.count("Zoya") << endl;   // 3.9 0
    cout << "load factor: " << cgpa.load_factor() << endl;

    unordered_set<int> seen = {1, 2, 3};
    cout << (seen.find(2) != seen.end()) << endl;               // 1

    // A tiny chaining table by hand
    vector<list<int>> table(5);
    for (int k : {15, 22, 7, 30, 12}) table[k % 5].push_back(k);
    for (int i = 0; i < 5; i++) {
        cout << i << ": ";
        for (int k : table[i]) cout << k << " -> ";
        cout << "NULL" << endl;
    }
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Word Frequency Report
Write `top_words(text, k)` that returns the `k` most frequent words (lower-case, punctuation `.,!?` removed) as a list of `(word, count)` tuples, sorted by count descending and then alphabetically. Use a dictionary (do not use `Counter`).

```text
top_words("Data is data. Data is power!", 2) → [("data", 3), ("is", 2)]
```

In [ ]:
def top_words(text, k):
    # TODO: clean, count with a dict, sort by (-count, word)
    pass

In [ ]:
# Test cell: run after your solution
assert top_words("Data is data. Data is power!", 2) == [("data", 3), ("is", 2)]
assert top_words("b a c a b", 3) == [("a", 2), ("b", 2), ("c", 1)]
assert top_words("", 3) == []
print("✅ Task 1 passed")

### Task 2 🟢 Linear Probing Hash Set
Complete `ProbingSet(capacity)` with `add(key)`, `contains(key)` and `remove(key)` using **open addressing with linear probing** and a `DELETED` tombstone. Assume the table never becomes full (no resizing needed).

```text
add 10, 17, 24 (capacity 7) → slots 3, 4, 5; remove 17; contains(24) → True
```

In [ ]:
class ProbingSet:
    DELETED = object()

    def __init__(self, capacity=11):
        self.slots = [None] * capacity

    def _probe(self, key):
        i = hash(key) % len(self.slots)
        while True:
            yield i
            i = (i + 1) % len(self.slots)

    def add(self, key):
        # TODO: if not already present, put key in the first None/DELETED slot
        pass

    def contains(self, key):
        # TODO: follow the probe sequence until None
        pass

    def remove(self, key):
        # TODO: replace with DELETED (do not set to None!)
        pass

In [ ]:
# Test cell: run after your solution
s = ProbingSet(7)
for k in [10, 17, 24]:
    s.add(k)
assert s.slots[3:6] == [10, 17, 24]
assert s.remove(17) is True and s.contains(24) is True and s.contains(17) is False
s.add(31); assert s.slots[4] == 31 and s.contains(31)
s.add(10); assert s.slots.count(10) == 1
assert s.remove(99) is False
print("✅ Task 2 passed")

### Task 3 🟡 Group Anagrams
Group words that are anagrams of each other. Use a dictionary whose key is a **canonical form** of the word (its sorted letters as a tuple or string). Return the groups sorted for easy checking: sort each group, then sort the list of groups.

```text
["eat", "tea", "tan", "ate", "nat", "bat"] → [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]
```

In [ ]:
def group_anagrams(words):
    groups = {}
    # TODO: canonical key = sorted letters
    return sorted(sorted(g) for g in groups.values())

In [ ]:
# Test cell: run after your solution
assert group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]
assert group_anagrams([]) == []
assert group_anagrams(["a"]) == [["a"]]
print("✅ Task 3 passed")

### Task 4 🟡 Complete the Hash Map
Extend `HashMap` from Part C into `MyMap` with: `__contains__(key)` (so `key in m` works), `keys()` (list of all keys), `__getitem__`/`__setitem__` (so `m[k]` works; missing keys raise `KeyError`) and `load_factor()`.

Test it with 1,000 keys and check that the load factor stays at or below 0.75.

In [ ]:
class MyMap(HashMap):
    _MISSING = object()

    def __contains__(self, key):
        # TODO
        pass

    def keys(self):
        # TODO
        pass

    def __getitem__(self, key):
        # TODO: raise KeyError if missing
        pass

    def __setitem__(self, key, value):
        # TODO
        pass

    def load_factor(self):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
m = MyMap()
for i in range(1000):
    m[f"roll{i}"] = i
assert m["roll500"] == 500 and "roll999" in m and "x" not in m
assert len(m.keys()) == 1000 and m.load_factor() <= 0.75
m["roll1"] = None; assert "roll1" in m and m["roll1"] is None
try:
    m["missing"]; raise AssertionError("should raise KeyError")
except KeyError:
    pass
print("✅ Task 4 passed")

### Task 5 🟡 Longest Consecutive Sequence
Given unsorted roll numbers, find the length of the **longest run of consecutive numbers** in **O(n)** (sorting would be O(n log n)).

Hint: put everything in a set. A number starts a run only if `x − 1` is not in the set; from each start, count upwards.

```text
[100, 4, 200, 1, 3, 2] → 4   (1, 2, 3, 4)
```

In [ ]:
def longest_consecutive(nums):
    s = set(nums)
    best = 0
    # TODO
    return best

In [ ]:
# Test cell: run after your solution
assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive([]) == 0
assert longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
assert longest_consecutive(list(range(0, 200_000, 2))) == 1
print("✅ Task 5 passed")

### Task 6 🔴 Subarray Sum Equals K (Prefix Sums + Hashing)
A shop’s daily profit/loss values are given (can be negative). Count how many **contiguous** periods have a total exactly equal to `k`.

Hint: keep a running prefix sum and a dictionary counting how many times each prefix sum has occurred. A period ending today has sum k if `prefix − k` occurred before. This is O(n) instead of O(n²).

```text
[1, 1, 1], k = 2 → 2
[3, 4, 7, 2, -3, 1, 4, 2], k = 7 → 4
```

In [ ]:
def count_subarrays(nums, k):
    seen = {0: 1}
    prefix = count = 0
    # TODO
    return count

In [ ]:
# Test cell: run after your solution
import random
def brute(a, k):
    return sum(1 for i in range(len(a)) for j in range(i, len(a)) if sum(a[i:j + 1]) == k)
assert count_subarrays([1, 1, 1], 2) == 2
assert count_subarrays([3, 4, 7, 2, -3, 1, 4, 2], 7) == 4
for _ in range(20):
    r = [random.randint(-3, 3) for _ in range(25)]
    assert count_subarrays(r, 2) == brute(r, 2)
print("✅ Task 6 passed")

### Task 7 🔴 Detect Plagiarism with Rolling Hashes
Two students submitted assignments. Find the **length of the longest common substring** of the two strings using **binary search on the length** and a **rolling (polynomial) hash** set of all substrings of that length (Rabin–Karp idea).

To keep things simple and safe, after finding hash matches, confirm by comparing the actual substrings.

```text
longest_common_substring("datastructures", "structural") → 8   ("structur")
```

In [ ]:
def longest_common_substring(a, b):
    MOD, BASE = (1 << 61) - 1, 131

    def substr_hashes(s, L):
        # TODO: return {hash: [start indexes]} for all substrings of length L (rolling hash)
        pass

    def common(L):
        # TODO: is there a substring of length L present in both a and b?
        pass

    # TODO: binary search the largest L for which common(L) is True
    return 0

In [ ]:
# Test cell: run after your solution
import random
def brute(a, b):
    best = 0
    for i in range(len(a)):
        for j in range(i + 1, len(a) + 1):
            if j - i > best and a[i:j] in b:
                best = j - i
    return best
assert longest_common_substring("datastructures", "structural") == 8
assert longest_common_substring("abc", "xyz") == 0
assert longest_common_substring("", "abc") == 0
for _ in range(20):
    x = "".join(random.choice("ab") for _ in range(25)); y = "".join(random.choice("ab") for _ in range(25))
    assert longest_common_substring(x, y) == brute(x, y)
print("✅ Task 7 passed")

## 🔬 Bonus: Watch the Chains Grow (Good vs Bad Hash)

In [ ]:
# Bonus: run after completing the tasks
words = [w + str(i) for i in range(2000) for w in ["roll"]]
m = 101

def chain_lengths(hash_fn):
    buckets = [0] * m
    for w in words:
        buckets[hash_fn(w) % m] += 1
    return max(buckets), sum(1 for b in buckets if b == 0)

bad = lambda s: sum(ord(c) for c in s)
good = lambda s: hash(s)
print("bad hash : longest chain = %d, empty buckets = %d" % chain_lengths(bad))
print("good hash: longest chain = %d, empty buckets = %d" % chain_lengths(good))
print("ideal average chain length =", round(len(words) / m, 1))

---
## 🎉 Lab 12 Complete!

Next: **Lab 13: Trees and Binary Trees**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab12/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)